# 🕵️ Dia 70 — Reconhecendo indicadores de aliciamento e tráfico em fontes publicadas

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 70* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-70.md) (*Recognizing grooming and trafficking indicators from published sources*), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Segunda, 26 de outubro. Você ia pesquisar o recrutador do relato E4 (vaga no exterior, visto de turista, destino vago). Rafa fecha a aba: *"Esse não é mais nosso. É indicador de tráfico. A gente encaminha."* No quadro: **190**, **Disque 100**, **SaferNet**.

Neste notebook você vai:

1. Transformar o **cartão de encaminhamento** numa função que respeita a ordem das perguntas.
2. Jogar o quiz **"pare e encaminhe" ou "prossiga"** com situações descritas de forma abstrata.
3. Montar a **matriz de fontes** com um validador.
4. Gerar o **cartão brasileiro** em mermaid e o registro do **encaminhamento do E4**.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Todos os dados são **fictícios** (domínios `.example`, telefones `+55 00 00000-00NN`, chaves Pix `PIXFICT-*`).

> ⚠️ **Sobre este notebook.** Ele é sobre o **procedimento de parada e encaminhamento**, não sobre reconhecer conversas. Não há aqui exemplos de mensagens, roteiros nem cenários com crianças ou adolescentes. As situações do quiz descrevem o que **o analista** encontra no trabalho, no nível de categoria, como o original faz.

## 1. O cartão de encaminhamento, em código

A ordem importa e é a do original: **perigo imediato primeiro**; a pergunta da **imagem** vem antes de qualquer denúncia, para que a regra de não copiar se aplique antes que alguém pense em evidência.

In [ ]:
# Destinos brasileiros (confira telefones e URLs no site de cada órgão antes de imprimir o seu cartão)
DESTINOS = {
    "emergencia": "190",
    "imagem": "SaferNet Brasil (denúncia anônima)",
    "crianca_em_risco": "SaferNet Brasil e Disque 100; Conselho Tutelar do município",
    "trafico": "Disque 100",
    "fraude": "Caminho de fraude do Dia 76 (banco/MED, boletim de ocorrência)",
}

def cartao_de_encaminhamento(perigo_imediato: bool,
                             possivel_imagem_de_crianca: bool,
                             suspeita_aliciamento_ou_sextorsao: bool,
                             suspeita_trafico: bool,
                             golpe_com_possiveis_traficados: bool):
    """
    Devolve a lista ORDENADA de passos. Cada passo é (tipo, texto):
    'PARE' = o que fazer/não fazer agora; 'ENCAMINHE' = para quem vai.
    """
    passos = []
    # 1) Perigo imediato: sempre a primeira pergunta, e a resposta não espera nada.
    if perigo_imediato:
        passos.append(("PARE", f"Chame {DESTINOS['emergencia']} primeiro. Todo o resto vem depois."))
    # 2) Imagem: a regra de NÃO COPIAR vem antes de qualquer ideia de evidência.
    if possivel_imagem_de_crianca:
        passos.append(("PARE", "Não baixe, não printe, não salve, não imprima, não encaminhe (nem a um colega)."))
        passos.append(("PARE", "Anote só URL, plataforma e nome de usuário. Saia da página."))
        passos.append(("ENCAMINHE", DESTINOS["imagem"]))
    # 3) Criança em risco on-line (aliciamento ou sextorsão).
    if suspeita_aliciamento_ou_sextorsao:
        passos.append(("ENCAMINHE", DESTINOS["crianca_em_risco"]))
    # 4) Tráfico, ou golpe que pode envolver trabalhadores traficados (Dia 67).
    if suspeita_trafico or golpe_com_possiveis_traficados:
        passos.append(("PARE", "Não confronte, não investigue mais, não alerte a pessoa, nada de 'justiceiros'."))
        passos.append(("ENCAMINHE", DESTINOS["trafico"]))
    if golpe_com_possiveis_traficados:
        passos.append(("ENCAMINHE", DESTINOS["fraude"] + " — além do Disque 100"))
    # 5) Nenhum dos ramos acima: é fraude comum, segue o caminho normal.
    if not passos:
        passos.append(("PROSSIGA", "Sem indicadores de parada. Siga o procedimento de fraude (Dia 76)."))
    return passos

# O relato E4 do Dia 67: adulto, vaga no exterior com indicadores de tráfico, sem perigo imediato conhecido.
for tipo, texto in cartao_de_encaminhamento(False, False, False, False, True):
    print(f"{tipo:10} {texto}")

> 💡 Repare que a função **não tem** um ramo "investigar mais para encaminhar melhor". Isso não é esquecimento: é o conteúdo do dia. Um indicador levanta uma pergunta; quem responde é quem tem autoridade e contexto.

## 2. 🎮 Quiz: "pare e encaminhe" ou "prossiga"?

Situações de trabalho de analista, descritas de forma abstrata. Responda **P** (pare e encaminhe) ou **S** (pode seguir o procedimento normal de fraude). O gabarito diz o ramo do cartão e o que **não** fazer.

Neste quiz, errar uma situação de parada não é "quase lá": a meta do Checkpoint BR é acertar **todas** as de parada. `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
MODO_JOGO = True

# (situação abstrata, é parada?, ramo do cartão, o que NÃO fazer)
SITUACOES = [
    ("Ao revisar o servidor exposto de um kit de phishing, uma pasta tem nome que sugere conteúdo envolvendo menores.",
     True, "Imagem → não copiar → SaferNet",
     "Não abrir a pasta, não baixar, não printar 'para provar'. Anote só a URL e saia."),
    ("Um relato de falso emprego oferece vaga no exterior com passagem e visto 'resolvidos', visto de turista e destino vago.",
     True, "Golpe com possíveis traficados → Disque 100 + caminho de fraude",
     "Não pesquisar o recrutador, não contatar quem fez o relato."),
    ("Um relato de golpe de tarefas pede 'taxa de liberação' por Pix para sacar um saldo fictício. Adulto, sem viagem.",
     False, "Fraude comum → Dia 76",
     "Siga a triagem de fraude normal (Dias 67 e 76)."),
    ("Um colega propõe criar um perfil se passando por adolescente para 'atrair' um suspeito e entregar à polícia.",
     True, "Nunca: se passar por alguém / grupo de 'flagrante'",
     "Não participar. Encaminhar o que já se sabe aos canais; o caminho lícito é programa oficial."),
    ("Uma denúncia descreve uma pessoa que não pode sair do local de trabalho e está sempre acompanhada por quem parece controlá-la.",
     True, "Tráfico → não confrontar → Disque 100",
     "Não ir ao local, não alertar a pessoa nem quem a acompanha."),
    ("No grafo do Dia 68, um telefone aparece em dois relatos de falso emprego de adultos, sem viagem.",
     False, "Fraude comum → grafo e Dia 76",
     "Continue com indicadores; não pesquise a pessoa por trás do número."),
    ("Durante a análise, você recebe a informação de que alguém está em risco agora.",
     True, "Perigo imediato → 190 primeiro",
     "Não terminar a análise antes de ligar."),
    ("Um relato de sextorsão contra um adolescente chega à caixa de denúncias da cooperativa.",
     True, "Criança em risco → SaferNet, Disque 100, Conselho Tutelar",
     "Não pedir as imagens 'para avaliar', não responder ao extorsionário."),
    ("Um e-mail de BEC pede mudança de chave Pix de um fornecedor. Nada envolve pessoas vulneráveis.",
     False, "Fraude comum → Dia 66 e Dia 76",
     "Siga a classificação de BEC e a ligação para o número cadastrado."),
    ("Um sistema de triagem automática marca um relato como 'possível tráfico' e sugere enviar alerta automático ao suspeito.",
     True, "Revisão humana treinada (Dia 73) + Disque 100",
     "Nunca acusação automática nem mensagem automática a qualquer envolvido."),
]

acertos_parada, total_parada, acertos = 0, sum(s[1] for s in SITUACOES), 0
for i, (situacao, parada, ramo, nao_fazer) in enumerate(SITUACOES, start=1):
    print(f"\n[{i:02d}] {situacao}")
    if MODO_JOGO:
        r = ""
        while r not in ("P", "S"):
            r = input("   P = pare e encaminhe / S = prossiga: ").strip().upper()
        certo = (r == "P") == parada
        acertos += certo
        acertos_parada += certo and parada
        print("   ✅" if certo else ("   ❌ Esta era de PARADA. Releia o cartão." if parada else "   ❌"))
    print(f"   Gabarito: {'PARE E ENCAMINHE' if parada else 'PROSSIGA'} — {ramo}")
    print(f"   Não fazer: {nao_fazer}")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(SITUACOES)} no total; {acertos_parada}/{total_parada} nas situações de parada.")
    print("Checkpoint BR cumprido." if acertos_parada == total_parada
          else "Checkpoint BR: refaça até acertar TODAS as de parada.")

## 3. A matriz de fontes, com validador

A matriz do original tem seis colunas. O validador aplica o checkpoint: toda linha tem **link**, tem a coluna **"sozinho, não pode dizer"** preenchida e **não nomeia pessoa** (indicador é categoria, como a fonte publica). As três linhas de exemplo são as do original; acrescente as suas na lista `MINHAS_LINHAS` até chegar a 15, a partir de pelo menos 4 fontes.

In [ ]:
import re

COLUNAS = ["indicador", "categoria", "publicado_por", "publico", "sozinho_nao_pode_dizer", "fonte"]

EXEMPLOS_DO_ORIGINAL = [
    {"indicador": "Adulto finge ser mais jovem ao contatar uma criança", "categoria": "Aliciamento on-line",
     "publicado_por": "NCMEC", "publico": "Pais, público",
     "sozinho_nao_pode_dizer": "Se o contato é um autor ou alguém mentindo por outro motivo; precisa de denúncia e revisão",
     "fonte": "https://www.missingkids.org/theissues/onlineenticement"},
    {"indicador": "A pessoa parece instruída sobre o que dizer", "categoria": "Tráfico",
     "publicado_por": "DHS Blue Campaign", "publico": "Público",
     "sozinho_nao_pode_dizer": "Se é coação, ansiedade, barreira de idioma ou medo da situação migratória",
     "fonte": "https://www.dhs.gov/blue-campaign/identify-victim"},
    {"indicador": "Recrutador oferece emprego no exterior com viagem e visto 'resolvidos' e função vaga",
     "categoria": "Tráfico para trabalho", "publicado_por": "OHCHR (2023)", "publico": "Formuladores de políticas, público",
     "sozinho_nao_pode_dizer": "Se é fraude, tráfico ou uma agência legítima mal administrada",
     "fonte": "https://bangkok.ohchr.org/sites/default/files/documents/2025-01/online-scam-operations-2582023.pdf"},
]
MINHAS_LINHAS = [
    # Acrescente aqui, no mesmo formato, parafraseando no nível de categoria da fonte.
]

def validar_matriz(linhas):
    problemas = []
    for i, l in enumerate(linhas, start=1):
        faltando = [c for c in COLUNAS if not l.get(c, "").strip()]
        if faltando:
            problemas.append(f"linha {i}: colunas vazias {faltando}")
        if not l.get("fonte", "").startswith("http"):
            problemas.append(f"linha {i}: fonte sem link (o checkpoint exige mostrar a frase da fonte)")
        # Heurística simples: arroba de usuário ou telefone indicam que saiu do nível de categoria
        if re.search(r"@\w+|\+\d[\d\s-]{7,}", l.get("indicador", "")):
            problemas.append(f"linha {i}: parece nomear alguém; indicador é categoria, não pessoa")
    fontes = {l.get("publicado_por", "").split(" (")[0] for l in linhas}
    return problemas, fontes

matriz = EXEMPLOS_DO_ORIGINAL + MINHAS_LINHAS
problemas, fontes = validar_matriz(matriz)
print(f"{len(matriz)} linhas, {len(fontes)} fontes: {sorted(fontes)}")
print("\n".join("❌ " + p for p in problemas) or "✅ Formato ok.")
print(f"Meta do original: 15+ linhas e 4+ fontes → faltam {max(0, 15 - len(matriz))} linhas e {max(0, 4 - len(fontes))} fontes.")

## 4. 📝 O cartão brasileiro e o encaminhamento do E4

In [ ]:
#@title Encaminhamento do relato E4 { display-mode: "form" }
ramo = "D5: golpe que pode envolver trabalhadores traficados"  #@param {type:"string"}
para_onde = "Disque 100, e o caminho de fraude do Dia 76 pela cooperativa"  #@param {type:"string"}
o_que_nao_fazemos = "Não pesquisamos o recrutador, não contatamos quem registrou o relato, não publicamos o número"  #@param {type:"string"}
quem_decide = "A autoridade competente"  #@param {type:"string"}

from pathlib import Path
from datetime import datetime, timezone

CARTAO_BR = """```mermaid
flowchart TD
    START(["O que eu vi?"]) --> D1{"Perigo imediato?"}
    D1 -- Sim --> EMS["190 primeiro"]
    EMS --> D2
    D1 -- Não --> D2{"Possível imagem sexual<br/>de criança ou adolescente?"}
    D2 -- Sim --> NOCOPY["Não baixe, não printe, não encaminhe"]
    NOCOPY --> NOTE["Anote URL, plataforma, usuário; saia"]
    NOTE --> CEI["SaferNet Brasil"]
    D2 -- Não --> D3{"Aliciamento ou sextorsão<br/>de criança ou adolescente?"}
    CEI --> D3
    D3 -- Sim --> CEE["SaferNet, Disque 100, Conselho Tutelar"]
    D3 -- Não --> D4{"Suspeita de tráfico?"}
    D4 -- Sim --> DNC["Não confronte, não investigue, não alerte"]
    DNC --> HT["Disque 100"]
    D4 -- Não --> D5{"Golpe com possíveis<br/>trabalhadores traficados?"}
    D5 -- Sim --> DNC
    D5 -- Sim --> FR["Caminho de fraude (Dia 76)"]
    D5 -- Não --> OTHER["Fraude comum (Dia 76)"]
```"""

registro = (
    "# Encaminhamento do relato E4 (Dia 70, Operação Isca, fictício)\n\n"
    f"- Registrado em (UTC): {datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}\n"
    f"- Ramo do cartão: {ramo}\n"
    f"- Para onde vai: {para_onde}\n"
    f"- O que a Vigia não faz: {o_que_nao_fazemos}\n"
    f"- Quem decide o resto: {quem_decide}\n\n"
    "## Cartão de encaminhamento (Brasil)\n\n" + CARTAO_BR + "\n\n"
    "Telefones e URLs do cartão impresso: conferir no site de cada órgão no dia da impressão.\n"
)

# O checkpoint pede no máximo 5 linhas no encaminhamento: contamos os itens.
itens = [l for l in registro.splitlines() if l.startswith("- ")]
print(f"Itens do encaminhamento: {len(itens)} (máximo 5)")

Path("notas").mkdir(exist_ok=True)
saida = Path("notas/dia70-encaminhamento-e4.md")
saida.write_text(registro, encoding="utf-8")
print(registro)
try:
    from google.colab import files
    files.download(str(saida))
except ImportError:
    print("(Fora do Colab: o arquivo ficou em notas/)")

### 🧠 Camada 8

O **impulso de resgate** parece virtude e é o que leva bons analistas a guardar o que não podiam guardar ou a alertar quem não deviam: é a justificação moral do Dia 1 no ponto de maior risco. E o **trauma vicário** é real para quem trabalha perto deste tema. A parada protege a vítima, a prova e você.

## ✅ Antes de fechar o Dia 70

- [ ] Todas as situações de **parada** do quiz certas
- [ ] Matriz com 15+ linhas, 4+ fontes, validador sem ❌
- [ ] Cartão (original e brasileiro) com emergência primeiro e o nó "não baixe nem encaminhe"
- [ ] `notas/dia70-encaminhamento-e4.md` com no máximo 5 itens e nenhuma pesquisa nova

Se este dia mexeu com você, faça uma pausa antes do Dia 71. No Brasil, o **CVV** atende pelo **188**, 24 horas, gratuitamente.

**Amanhã (Dia 71):** *deepfake* de voz, vídeo e imagem em fraudes. A checagem da chamada de vídeo do Dia 69, agora contra quem sabe trocar de rosto.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>